In [11]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [12]:
import random
from sklearn.model_selection import cross_val_score

In [13]:
intent_queries = {
"TOTAL_SALES": [
"Show me total sales", "What is overall revenue?", "How much did we sell in total?",
"Give me total earnings", "Calculate all sales", "What’s the total profit?",
"Display complete sales amount", "Revenue overall", "Sum of all sales",
"Give me sales summary", "How much did we earn?", "Total revenue please",
"Show overall turnover", "Total income of company", "What’s our gross sales?",
"Provide overall sales data", "What’s the total amount sold?", "Show me combined sales",
"Give me total revenue numbers", "Summarize all sales"
],
"TREND_MONTH": [
"Show me monthly sales trend", "Sales growth by month", "How sales changed month to month?",
"Plot monthly revenue trend", "Monthly trend of profits", "Compare months by sales",
"What is the trend in last months?", "Revenue variation each month", "Month wise performance",
"How are monthly earnings moving?", "Trend of sales per month", "Give line chart of month sales",
"Growth over months", "Month to month sales data", "Monthly increase or decrease",
"Track sales over months", "Which month performed best?", "Show changes month-wise",
"Graph of sales by months", "Month trend report"
],
"FILTER_REGION": [
"Show sales in North region", "Sales only for East region", "Give revenue for South",
"Filter data by region", "Show me sales for West zone", "Revenue in North only",
"Sales figures for East zone", "Region-specific performance", "Profit in South",
"Display West region results", "North zone report", "East zone revenue", "South area sales",
"Give me region-wise breakdown", "Region wise report of sales",
"Revenue from South zone", "What’s North region profit?", "Data for East only",
"Show only South sales", "Filter to West region"
],
"AVG_PROFIT_YEAR": [
"Show average yearly profit", "What is avg profit each year?", "Year wise average earnings",
"Calculate annual profit average", "Year average revenue", "How much on average per year?",
"Yearly avg turnover", "Give average of yearly sales", "Profit average by year",
"Display avg revenue per year", "Annual mean profit", "Avg sales in a year",
"What’s the yearly average income?", "Revenue average each year", "Year avg growth",
"Year-wise avg profit trend", "Calculate yearly averages", "Give mean of yearly profits",
"Show avg earning trend annually", "Summarize average profit per year"
],
"AGGREGATE_REGION": [
"Summarize sales by region", "Aggregate revenue across regions", "Group sales by zone",
"Show me sales total by area", "Region wise aggregation of sales", "Breakdown by regions",
"Revenue grouped per region", "Profit aggregated by region", "Regional totals of sales",
"Display sales grouped by geography", "Zone wise aggregation", "Combine sales by region",
"Total sales each region", "Overall sales grouped regionally", "Show aggregated numbers per region",
"Sales by territory", "Sum grouped by area", "Aggregate by regional divisions",
"Revenue distribution per region", "Summarize zones by sales"
],
"TOP_PRODUCTS": [
"Show top selling products", "Which product sold the most?", "Give me best performing items",
"Top 5 products by sales", "Highest revenue items", "Most popular products",
"List top 10 items", "Best sellers report", "Which products are trending?",
"Show products with highest profit", "Popular products list", "Top revenue generating items",
"Best 3 products by sales", "Highest demand items", "Show me the hot products",
"Top performers in sales", "Which item is best selling?", "Products with max sales",
"Best selling products overall", "Top sales products chart"
]
}

In [14]:
queries, labels = [], []
for label, qs in intent_queries.items():
    for q in qs:
        queries.append(q)
        labels.append(label)


# Create DataFrame
data = pd.DataFrame({"query": queries, "intent": labels})
print("Dataset size:", data.shape)

Dataset size: (120, 2)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(data["query"], data["intent"],
test_size=0.2, random_state=42, stratify=data["intent"])

In [16]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [17]:
y_pred = clf.predict(X_test_vec)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.875

Classification Report:
                   precision    recall  f1-score   support

AGGREGATE_REGION       0.67      0.50      0.57         4
 AVG_PROFIT_YEAR       1.00      1.00      1.00         4
   FILTER_REGION       0.60      0.75      0.67         4
    TOP_PRODUCTS       1.00      1.00      1.00         4
     TOTAL_SALES       1.00      1.00      1.00         4
     TREND_MONTH       1.00      1.00      1.00         4

        accuracy                           0.88        24
       macro avg       0.88      0.88      0.87        24
    weighted avg       0.88      0.88      0.87        24



In [18]:
scores = cross_val_score(clf, vectorizer.transform(data["query"]), data["intent"], cv=5)
print("\nCross-validation Accuracy (5-fold):", scores.mean())


Cross-validation Accuracy (5-fold): 0.925


In [19]:
def predict_with_confidence(user_query, threshold=0.5):
    vec = vectorizer.transform([user_query])
    probs = clf.predict_proba(vec).max()
    label = clf.predict(vec)[0]
    if probs < threshold:
        return "I didn’t understand, please rephrase."
    else:
        return label

In [20]:
# You can reuse your expanded dataset dict; add a few synonyms for robustness
intent_queries = {
    "TOTAL_SALES": [
        "Show me total sales","What is overall revenue?","How much did we sell in total?",
        "Give me total earnings","Calculate all sales","Sum of all sales","Total revenue please",
        "Overall turnover","Total income of company","Combined sales amount","Gross sales overall",
        "How much revenue did we make?"
    ],
    "TREND_MONTH": [
        "Show monthly sales trend","Sales growth by month","Plot monthly revenue trend",
        "Monthly trend of profits","Month wise performance","Line chart of month sales",
        "Month-to-month sales changes","Which month performed best?"
    ],
    "FILTER_REGION": [
        "Show sales in North region","Sales only for East region","Revenue for South region",
        "Profit in West region","Filter to North zone","East zone revenue","South area sales",
        "West region performance"
    ],
    "AVG_PROFIT_YEAR": [
        "Average profit in 2023","Show average yearly profit","Annual mean profit",
        "Year-wise average earnings","Avg sales in a year","Average revenue per year",
        "Yearly average profit"
    ],
    "AGGREGATE_REGION": [
        "Summarize sales by region","Total sales each region","Group sales by region",
        "Revenue grouped per region","Region wise aggregation of sales","Regional sales totals"
    ],
    "TOP_PRODUCTS": [
        "Top 5 products by sales","Which product sold the most?","Best selling products",
        "Highest revenue items","Popular products list","Top revenue generating items"
    ]
}

# Flatten to a DataFrame
rows = []
for intent, qs in intent_queries.items():
    for q in qs:
        rows.append({"query": q, "intent": intent})
train_df = pd.DataFrame(rows)
print("Training samples:", len(train_df))
train_df.sample(5)


Training samples: 47


,query,intent
34,Yearly average profit,AVG_PROFIT_YEAR
36,Total sales each region,AGGREGATE_REGION
3,Give me total earnings,TOTAL_SALES
26,South area sales,FILTER_REGION
12,Show monthly sales trend,TREND_MONTH


In [22]:
!pip install -q sentence-transformers


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
!pip install protobuf==3.20.3

   ---------------------------------------- 0.0/904.0 kB ? eta -:--:--
   ---------------------------------------- 904.0/904.0 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.31.1
    Uninstalling protobuf-6.31.1:
      Successfully uninstalled protobuf-6.31.1


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.10.1 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.10.1 requires protobuf<3.20,>=3.9.2, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip uninstall -y sentence-transformers
!pip install sentence-transformers==2.2.2


Found existing installation: sentence-transformers 5.1.0
Uninstalling sentence-transformers-5.1.0:
  Successfully uninstalled sentence-transformers-5.1.0
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 6.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------------- -------------------- 0.8/1.6 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/241.4 MB ? eta -:--:--
   ---------------------------------------- 0.8/241.4 MB 3.7 MB/s eta 0:01:05
   ---------------------------------------- 1.6/241.4 MB 4.0 MB/s eta 0:01:01
   ---------------------------------------- 2.4/241.4 MB 3.8 MB/s eta 0:01:03
    --------------------------------------- 3.1/241.4 MB 3.9 MB/s eta 0:0

  DEPRECATION: Building 'sentence-transformers' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sentence-transformers'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  You can safely remove it manually.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install torch==2.1.0


   ---------------------------------------- 0.0/192.3 MB ? eta -:--:--
   ---------------------------------------- 1.0/192.3 MB 6.3 MB/s eta 0:00:31
   ---------------------------------------- 1.8/192.3 MB 4.6 MB/s eta 0:00:42
    --------------------------------------- 2.6/192.3 MB 4.3 MB/s eta 0:00:44
    --------------------------------------- 3.4/192.3 MB 4.2 MB/s eta 0:00:46
    --------------------------------------- 4.5/192.3 MB 4.1 MB/s eta 0:00:47
   - -------------------------------------- 5.2/192.3 MB 4.0 MB/s eta 0:00:47
   - -------------------------------------- 6.0/192.3 MB 4.1 MB/s eta 0:00:46
   - -------------------------------------- 6.8/192.3 MB 4.1 MB/s eta 0:00:46
   - -------------------------------------- 7.6/192.3 MB 4.0 MB/s eta 0:00:47
   - -------------------------------------- 8.4/192.3 MB 4.0 MB/s eta 0:00:46
   - -------------------------------------- 9.2/192.3 MB 4.0 MB/s eta 0:00:46
   -- ------------------------------------- 10.0/192.3 MB 4.0 MB/s eta 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.23.0 requires torch==2.8.0, but you have torch 2.1.0 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install --upgrade torch==2.8.0 torchvision==0.23.0
!pip install --upgrade sentence-transformers protobuf==3.20.3


  Using cached torch-2.8.0-cp310-cp310-win_amd64.whl.metadata (30 kB)
Using cached torch-2.8.0-cp310-cp310-win_amd64.whl (241.4 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.1.0
    Uninstalling torch-2.1.0:
      Successfully uninstalled torch-2.1.0



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.0-py3-none-any.whl (483 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [21]:
intent_queries = {
    "TOTAL_SALES": [
        "Show me total sales","What is overall revenue?","How much did we sell in total?",
        "Give me total earnings","Calculate all sales","Sum of all sales","Total revenue please",
        "Overall turnover","Total income of company","Combined sales amount","Gross sales overall",
        "How much revenue did we make?"
    ],
    "TREND_MONTH": [
        "Show monthly sales trend","Sales growth by month","Plot monthly revenue trend",
        "Monthly trend of profits","Month wise performance","Line chart of month sales",
        "Month-to-month sales changes","Which month performed best?"
    ],
    "FILTER_REGION": [
        "Show sales in North region","Sales only for East region","Revenue for South region",
        "Profit in West region","Filter to North zone","East zone revenue","South area sales",
        "West region performance"
    ],
    "AVG_PROFIT_YEAR": [
        "Average profit in 2023","Show average yearly profit","Annual mean profit",
        "Year-wise average earnings","Avg sales in a year","Average revenue per year",
        "Yearly average profit"
    ],
    "AGGREGATE_REGION": [
        "Summarize sales by region","Total sales each region","Group sales by region",
        "Revenue grouped per region","Region wise aggregation of sales","Regional sales totals"
    ],
    "TOP_PRODUCTS": [
        "Top 5 products by sales","Which product sold the most?","Best selling products",
        "Highest revenue items","Popular products list","Top revenue generating items"
    ]
}

# Flatten to DataFrame
rows = []
for intent, qs in intent_queries.items():
    for q in qs:
        rows.append({"query": q, "intent": intent})
train_df = pd.DataFrame(rows)


In [22]:
vectorizer = TfidfVectorizer()
train_texts = train_df["query"].tolist()
train_labels = train_df["intent"].tolist()
train_emb = vectorizer.fit_transform(train_texts)  # sparse TF-IDF matrix


In [23]:
def predict_intent_tfidf(user_query: str, threshold: float = 0.2):
    q_emb = vectorizer.transform([user_query])
    sims = cosine_similarity(train_emb, q_emb).ravel()
    idx = np.argmax(sims)
    best_sim = sims[idx]
    best_intent = train_labels[idx]
    best_example = train_texts[idx]
    
    if best_sim < threshold:
        return None, best_sim, best_example
    return best_intent, best_sim, best_example


In [24]:
tests = [
    "What was our revenue last year?",
    "Give overall sales",
    "Show me region wise totals",
    "Monthly revenue pattern",
    "Best performing items"
]

for t in tests:
    intent, sim, match = predict_intent_tfidf(t, threshold=0.2)
    print(f"Q: {t}\n → intent: {intent} | similarity: {sim:.2f} | matched: {match}\n")


Q: What was our revenue last year?
 → intent: TOTAL_SALES | similarity: 0.54 | matched: What is overall revenue?

Q: Give overall sales
 → intent: TOTAL_SALES | similarity: 0.47 | matched: Gross sales overall

Q: Show me region wise totals
 → intent: TOTAL_SALES | similarity: 0.53 | matched: Show me total sales

Q: Monthly revenue pattern
 → intent: TREND_MONTH | similarity: 0.62 | matched: Plot monthly revenue trend

Q: Best performing items
 → intent: TOP_PRODUCTS | similarity: 0.43 | matched: Highest revenue items



In [31]:
import re
import pandas as pd

def run_intent_on_dataframe(intent, df, user_query=None):
    if intent == "TOTAL_SALES":
        return df["Sales"].sum()
    
    if intent == "TREND_MONTH":
        tmp = df.copy()
        tmp["Month"] = pd.to_datetime(df["Date"]).dt.to_period("M").astype(str)
        
        #detect month/year from query
        if user_query:
            # Detect year like 2024
            year_match = re.search(r"\b(20\d{2})\b", user_query)
            if year_match:
                tmp = tmp[pd.to_datetime(tmp["Date"]).dt.year == int(year_match.group(1))]
        return tmp.groupby("Month")["Sales"].sum().sort_index()
    
    if intent == "FILTER_REGION":
        regions = ["North", "South", "East", "West"]
        if user_query:
            region = next((r for r in regions if r.lower() in user_query.lower()), "East")
        else:
            region = "East"
        return df[df["Region"] == region][["Region","Sales","Profit"]]
    
    if intent == "AVG_PROFIT_YEAR":
        tmp = df.copy()
        tmp["Year"] = pd.to_datetime(df["Date"]).dt.year
        if user_query:
            year_match = re.search(r"\b(20\d{2})\b", user_query)
            if year_match:
                tmp = tmp[tmp["Year"] == int(year_match.group(1))]
        return tmp.groupby("Year")["Profit"].mean()
    
    if intent == "AGGREGATE_REGION":
        return df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
    
    if intent == "TOP_PRODUCTS":
        tmp = df.groupby("Product")["Sales"].sum().sort_values(ascending=False)
        if user_query:
            n_match = re.search(r"\btop (\d+)\b", user_query.lower())
            if n_match:
                n = int(n_match.group(1))
                return tmp.head(n)
        return tmp.head(5)
    
    return "Intent not mapped yet."


In [27]:
import pandas as pd
df = pd.read_excel("sales_data.xlsx")  


df.head()


,Date,Region,Sales,Profit,Product
0,2024-05-20 08:48:58.776,East,1576,389,Tablet
1,2024-04-13 10:17:08.571,West,4493,847,Laptop
2,2024-02-21 17:08:34.286,South,879,380,Mobile
3,2024-06-04 03:25:42.857,South,992,689,Mobile
4,2024-01-26 20:34:17.143,North,2562,555,Monitor


In [32]:
user_query = "Show sales in North region"

# Use your TF-IDF intent prediction
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.20)

if intent is None:
    print("I didn’t understand, please rephrase.")
else:
    print(f"Intent: {intent} (sim {sim:.2f}, matched: '{match}')")
    result = run_intent_on_dataframe(intent, df)
    display(result)

Intent: FILTER_REGION (sim 1.00, matched: 'Show sales in North region')


,Region,Sales,Profit
0,East,1576,389
8,East,2527,239
13,East,4061,682
25,East,4340,868
42,East,3814,559
45,East,3229,436
47,East,3560,662


In [34]:
result = run_intent_on_dataframe(intent, df, user_query=user_query)
display(result)


,Region,Sales,Profit
4,North,2562,555
6,North,564,522
10,North,1995,434
11,North,891,426
26,North,1528,283
35,North,1063,741
37,North,2257,879
40,North,1559,613
49,North,4127,130


In [35]:
user_query = "Give me total sales"
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.50)

if intent is None:
    print("I didn’t understand, please rephrase.")
else:
    print(f"Intent: {intent} (sim {sim:.2f}, matched: '{match}')")
    result = run_intent_on_dataframe(intent, df, user_query)
    print("Result:\n", result)


Intent: TOTAL_SALES (sim 0.82, matched: 'Give me total earnings')
Result:
 137087


In [36]:
user_query = "Show monthly sales trend"
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.50)

if intent is not None:
    result = run_intent_on_dataframe(intent, df, user_query)
    print("Monthly Trend:\n", result)


Monthly Trend:
 Month
2024-01    27577
2024-02    17159
2024-03    34338
2024-04    20086
2024-05    21104
2024-06    16823
Name: Sales, dtype: int64


In [37]:
user_query = "Show average yearly profit"
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.50)

if intent is not None:
    result = run_intent_on_dataframe(intent, df, user_query)
    print("Average Profit per Year:\n", result)


Average Profit per Year:
 Year
2024    523.24
Name: Profit, dtype: float64


In [38]:
user_query = "Summarize sales by region"
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.50)

if intent is not None:
    result = run_intent_on_dataframe(intent, df, user_query)
    print("Sales by Region:\n", result)


Sales by Region:
 Region
West     58718
South    38716
East     23107
North    16546
Name: Sales, dtype: int64


In [39]:
user_query = "Top 5 products by sales"
intent, sim, match = predict_intent_tfidf(user_query, threshold=0.50)

if intent is not None:
    result = run_intent_on_dataframe(intent, df, user_query)
    print("Top Products:\n", result)


Top Products:
 Product
Monitor       36282
Mobile        29364
Tablet        29013
Laptop        28105
Headphones    14323
Name: Sales, dtype: int64
